# 🫀 ECG Multi-Label Classification — Kaggle Training (All Models)

Self-contained notebook that trains **all 8 models** sequentially on Kaggle GPU.  
After training, download `outputs/` and resume locally.

**Dataset**: Add `ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1` as a Kaggle dataset.

**Models trained (in order)**:
1. CNN 1D
2. Leadwise CNN
3. ResNet 1D
4. Pretrained ResNet 1D (xresnet-D)
5. LSTM (Bidirectional + Attention)
6. Transformer
7. CNN-LSTM
8. CNN-Transformer

## 1. Setup & Install

In [ ]:
!pip install -q wfdb scikit-learn scipy

import ast, json, logging, math, os, random, shutil, time, gc
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from scipy.signal import butter, filtfilt
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, average_precision_score)
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau, StepLR
from torch.utils.data import DataLoader, Dataset
import wfdb

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f'PyTorch {torch.__version__} | Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    if torch.cuda.device_count() > 1:
        print(f'Multi-GPU: {torch.cuda.device_count()} GPUs available')

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════
#  DATA PATH — update to your Kaggle dataset path
# ═══════════════════════════════════════════════════════════
DATA_DIR = "/kaggle/input/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.1"

# ═══════════════════════════════════════════════════════════
#  TRAINING ORDER — comment out any model to skip it
# ═══════════════════════════════════════════════════════════
MODELS_TO_TRAIN = [
    "cnn_1d",
    "leadwise_cnn",
    "resnet",
    "pretrained_resnet",
    "lstm",
    "transformer",
    "cnn_lstm",
    "cnn_transformer",
]

# ═══════════════════════════════════════════════════════════
#  Model-specific hyperparameters
# ═══════════════════════════════════════════════════════════
MODEL_CONFIGS = {
    "cnn_1d": dict(
        params=dict(base_filters=64, n_blocks=4, kernel_size=7, dropout=0.3),
        batch_size=128, lr=1e-3, epochs=50, patience=5,
    ),
    "leadwise_cnn": dict(
        params=dict(base_filters=32, n_blocks=3, kernel_size=7, dropout=0.3),
        batch_size=64, lr=1e-3, epochs=60, patience=5,
    ),
    "resnet": dict(
        params=dict(base_filters=64, n_blocks_per_stage=2, kernel_size=7, dropout=0.3),
        batch_size=64, lr=5e-4, epochs=80, patience=5,
    ),
    "pretrained_resnet": dict(
        params=dict(base_filters=64, layers=[2,2,2,2], kernel_size=7, dropout=0.2),
        batch_size=64, lr=5e-4, epochs=80, patience=5,
    ),
    "lstm": dict(
        params=dict(hidden_size=128, num_layers=2, dropout=0.3),
        batch_size=64, lr=1e-3, epochs=50, patience=5,
    ),
    "transformer": dict(
        params=dict(d_model=128, nhead=8, num_layers=4, dim_feedforward=256, dropout=0.1),
        batch_size=64, lr=3e-4, epochs=60, patience=5,
    ),
    "cnn_lstm": dict(
        params=dict(cnn_filters=64, n_cnn_blocks=3, kernel_size=7, lstm_hidden=128, lstm_layers=2, dropout=0.3),
        batch_size=64, lr=1e-3, epochs=50, patience=5,
    ),
    "cnn_transformer": dict(
        params=dict(cnn_filters=64, n_cnn_blocks=3, kernel_size=7, d_model=128, nhead=8,
                    num_transformer_layers=3, dim_feedforward=256, dropout=0.1),
        batch_size=64, lr=3e-4, epochs=60, patience=5,
    ),
}

print(f"Will train {len(MODELS_TO_TRAIN)} models: {', '.join(MODELS_TO_TRAIN)}")

## 3. Data Pipeline

In [ ]:
# ── Label constants ──
DIAGNOSTIC_SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

def load_metadata(data_dir):
    df = pd.read_csv(Path(data_dir) / "ptbxl_database.csv", index_col="ecg_id")
    df.scp_codes = df.scp_codes.apply(ast.literal_eval)
    return df

def load_scp_statements(data_dir):
    return pd.read_csv(Path(data_dir) / "scp_statements.csv", index_col=0)

def load_raw_signals(metadata_df, data_dir, sampling_rate=100):
    data_path = str(Path(data_dir)) + "/"
    filenames = metadata_df.filename_lr if sampling_rate == 100 else metadata_df.filename_hr
    signals = [wfdb.rdsamp(data_path + f)[0] for f in filenames]
    return np.array(signals, dtype=np.float32)

def aggregate_diagnostics(metadata_df, scp_df, label_type="diagnostic_superclass"):
    diag_df = scp_df[scp_df.diagnostic == 1.0]
    col = "diagnostic_class" if label_type == "diagnostic_superclass" else "diagnostic_subclass"
    def _agg(scp_dict):
        return list(set(
            diag_df.loc[c][col] for c in scp_dict if c in diag_df.index and pd.notna(diag_df.loc[c][col])
        ))
    return metadata_df.scp_codes.apply(_agg)

def encode_labels(diag_labels, label_classes=None):
    if label_classes is None:
        label_classes = DIAGNOSTIC_SUPERCLASSES
    idx_map = {c: i for i, c in enumerate(label_classes)}
    matrix = np.zeros((len(diag_labels), len(label_classes)), dtype=np.float32)
    for i, labels in enumerate(diag_labels):
        for l in labels:
            if l in idx_map:
                matrix[i, idx_map[l]] = 1.0
    return matrix, label_classes

def compute_class_weights(label_matrix):
    counts = np.where(label_matrix.sum(0) == 0, 1, label_matrix.sum(0))
    return (label_matrix.shape[0] / (label_matrix.shape[1] * counts)).astype(np.float32)

In [ ]:
# ── Preprocessing ──

def bandpass_filter(signal, lowcut=0.5, highcut=40.0, fs=100.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    axis = 0 if signal.ndim == 2 else 1
    return filtfilt(b, a, signal, axis=axis).astype(np.float32)

def remove_baseline_wander(signal, fs=100.0, cutoff=0.5, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff/nyq, btype='high')
    axis = 0 if signal.ndim == 2 else 1
    return filtfilt(b, a, signal, axis=axis).astype(np.float32)

def preprocess_pipeline(signals, config=None):
    signals = remove_baseline_wander(signals)
    signals = bandpass_filter(signals)
    high = np.percentile(np.abs(signals), 99)
    signals = np.clip(signals, -high, high).astype(np.float32)
    mean = signals.mean(axis=1, keepdims=True)
    std = np.where(signals.std(axis=1, keepdims=True) == 0, 1.0, signals.std(axis=1, keepdims=True))
    signals = ((signals - mean) / std).astype(np.float32)
    print(f"Preprocessed: {signals.shape}")
    return signals

class ECGDataset(Dataset):
    def __init__(self, signals, labels):
        self.signals = signals
        self.labels = labels
    def __len__(self):
        return len(self.signals)
    def __getitem__(self, idx):
        return (torch.tensor(self.signals[idx], dtype=torch.float32).permute(1, 0),
                torch.tensor(self.labels[idx], dtype=torch.float32))

## 4. Load & Preprocess Data (once)

In [ ]:
metadata = load_metadata(DATA_DIR)
signals = load_raw_signals(metadata, DATA_DIR, sampling_rate=100)
signals = preprocess_pipeline(signals)

scp_df = load_scp_statements(DATA_DIR)
diag_labels = aggregate_diagnostics(metadata, scp_df)
label_matrix, label_classes = encode_labels(diag_labels)
class_weights = compute_class_weights(label_matrix)

# Splits
folds = metadata.strat_fold.values
train_idx = np.where((folds != 9) & (folds != 10))[0]
val_idx = np.where(folds == 9)[0]
test_idx = np.where(folds == 10)[0]

print(f"Samples — Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print(f"Classes: {label_classes}")
print(f"Weights: {class_weights.round(3)}")

## 5. Model Definitions

In [ ]:
# ── CNN 1D ──
class CNN1D(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, base_filters=64, n_blocks=4, kernel_size=7, dropout=0.3, **kw):
        super().__init__()
        blocks, in_ch = [], input_channels
        for i in range(n_blocks):
            out_ch = base_filters * (2**i)
            blocks.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size//2),
                nn.BatchNorm1d(out_ch), nn.ReLU(True), nn.MaxPool1d(2, 2)))
            in_ch = out_ch
        self.features = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(in_ch, num_classes)
    def forward(self, x):
        return self.fc(self.dropout(self.pool(self.features(x)).squeeze(-1)))

# ── Leadwise CNN ──
class LeadwiseCNN(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, base_filters=32, n_blocks=3, kernel_size=7, dropout=0.3, **kw):
        super().__init__()
        self.n_leads = input_channels
        layers, in_ch = [], 1
        for i in range(n_blocks):
            out_ch = base_filters * (2**i)
            layers.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size//2),
                nn.BatchNorm1d(out_ch), nn.ReLU(True), nn.MaxPool1d(2, 2)))
            in_ch = out_ch
        self.backbone = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(in_ch * self.n_leads, 128),
            nn.ReLU(True), nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x):
        feats = [self.pool(self.backbone(x[:, i:i+1, :])).squeeze(-1) for i in range(self.n_leads)]
        return self.classifier(torch.cat(feats, 1))

# ── ResNet 1D ──
class ResidualBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1):
        super().__init__()
        pad = kernel_size // 2
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(True)
        self.skip = (nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, stride), nn.BatchNorm1d(out_ch))
                     if stride != 1 or in_ch != out_ch else nn.Identity())
    def forward(self, x):
        return self.relu(self.bn2(self.conv2(self.relu(self.bn1(self.conv1(x))))) + self.skip(x))

class ResNet1D(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, base_filters=64, n_blocks_per_stage=2, kernel_size=7, dropout=0.3, **kw):
        super().__init__()
        bf = base_filters
        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, bf, 15, stride=2, padding=7), nn.BatchNorm1d(bf),
            nn.ReLU(True), nn.MaxPool1d(3, 2, 1))
        def stage(ic, oc, n, ks, s):
            return nn.Sequential(ResidualBlock1D(ic, oc, ks, s), *[ResidualBlock1D(oc, oc, ks) for _ in range(n-1)])
        self.s1 = stage(bf, bf, n_blocks_per_stage, kernel_size, 1)
        self.s2 = stage(bf, bf*2, n_blocks_per_stage, kernel_size, 2)
        self.s3 = stage(bf*2, bf*4, n_blocks_per_stage, kernel_size, 2)
        self.s4 = stage(bf*4, bf*8, n_blocks_per_stage, kernel_size, 2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(bf*8, num_classes)
    def forward(self, x):
        x = self.s4(self.s3(self.s2(self.s1(self.stem(x)))))
        return self.fc(self.dropout(self.pool(x).squeeze(-1)))

# ── Pretrained ResNet 1D (xresnet-D) ──
class PretrainedResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size=7, stride=1, dropout=0.1):
        super().__init__()
        pad = kernel_size // 2
        self.bn1 = nn.BatchNorm1d(in_ch)
        self.relu1 = nn.ReLU(True)
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size, stride=stride, padding=pad)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu2 = nn.ReLU(True)
        self.do = nn.Dropout(dropout)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size, padding=pad)
        self.needs_proj = (stride != 1 or in_ch != out_ch)
        self.skip = (nn.Sequential(nn.Conv1d(in_ch, out_ch, 1, stride=stride), nn.BatchNorm1d(out_ch))
                     if self.needs_proj else nn.Identity())
    def forward(self, x):
        identity = self.skip(x)
        out = self.conv1(self.relu1(self.bn1(x)))
        out = self.conv2(self.do(self.relu2(self.bn2(out))))
        if out.shape[-1] != identity.shape[-1]:
            ml = min(out.shape[-1], identity.shape[-1])
            out, identity = out[..., :ml], identity[..., :ml]
        return out + identity

class PretrainedResNet1D(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, base_filters=64, layers=None,
                 kernel_size=7, dropout=0.2, **kw):
        super().__init__()
        if layers is None: layers = [2,2,2,2]
        bf = base_filters
        self.stem = nn.Sequential(
            nn.Conv1d(input_channels, bf//2, 7, stride=2, padding=3), nn.BatchNorm1d(bf//2), nn.ReLU(True),
            nn.Conv1d(bf//2, bf//2, 3, padding=1), nn.BatchNorm1d(bf//2), nn.ReLU(True),
            nn.Conv1d(bf//2, bf, 3, padding=1), nn.BatchNorm1d(bf), nn.ReLU(True),
            nn.MaxPool1d(3, 2, 1))
        fs = [bf, bf*2, bf*4, bf*8]
        def stg(ic, oc, n, s):
            return nn.Sequential(PretrainedResBlock(ic, oc, kernel_size, s, dropout),
                                 *[PretrainedResBlock(oc, oc, kernel_size, 1, dropout) for _ in range(n-1)])
        self.stage1 = stg(bf, fs[0], layers[0], 1)
        self.stage2 = stg(fs[0], fs[1], layers[1], 2)
        self.stage3 = stg(fs[1], fs[2], layers[2], 2)
        self.stage4 = stg(fs[2], fs[3], layers[3], 2)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(fs[3], num_classes))
    def forward(self, x):
        x = self.stage4(self.stage3(self.stage2(self.stage1(self.stem(x)))))
        return self.head(self.pool(x).squeeze(-1))
    def save_backbone(self, path):
        torch.save({k: v for k, v in self.state_dict().items() if not k.startswith("head.")}, path)

# ── LSTM ──
class LSTMModel(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, hidden_size=128, num_layers=2, dropout=0.3, **kw):
        super().__init__()
        self.lstm = nn.LSTM(input_channels, hidden_size, num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        dim = hidden_size * 2
        self.attn = nn.Sequential(nn.Linear(dim, 64), nn.Tanh(), nn.Linear(64, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(dim, num_classes)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        out, _ = self.lstm(x)
        w = torch.softmax(self.attn(out), dim=1)
        return self.fc(self.dropout((out * w).sum(1)))

# ── Transformer ──
class TransformerModel(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, d_model=128, nhead=8,
                 num_layers=4, dim_feedforward=256, dropout=0.1, max_seq_len=5000, **kw):
        super().__init__()
        self.proj = nn.Linear(input_channels, d_model)
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(max_seq_len).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.pe_drop = nn.Dropout(dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, activation='gelu')
        self.encoder = nn.TransformerEncoder(layer, num_layers)
        self.fc = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, num_classes))
    def forward(self, x):
        x = self.proj(x.permute(0, 2, 1))
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], 1)
        x = self.pe_drop(x + self.pe[:, :x.size(1)])
        return self.fc(self.encoder(x)[:, 0])

# ── CNN-LSTM ──
class CNNLSTM(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, cnn_filters=64, n_cnn_blocks=3,
                 kernel_size=7, lstm_hidden=128, lstm_layers=2, dropout=0.3, **kw):
        super().__init__()
        layers, in_ch = [], input_channels
        for i in range(n_cnn_blocks):
            out_ch = cnn_filters * (2**i)
            layers.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size//2),
                nn.BatchNorm1d(out_ch), nn.ReLU(True), nn.MaxPool1d(2, 2)))
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        self.lstm = nn.LSTM(in_ch, lstm_hidden, lstm_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if lstm_layers > 1 else 0.0)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden * 2, num_classes)
    def forward(self, x):
        x = self.cnn(x).permute(0, 2, 1)
        _, (h, _) = self.lstm(x)
        return self.fc(self.dropout(torch.cat([h[-2], h[-1]], 1)))

# ── CNN-Transformer ──
class CNNTransformer(nn.Module):
    def __init__(self, input_channels=12, num_classes=5, cnn_filters=64, n_cnn_blocks=3,
                 kernel_size=7, d_model=128, nhead=8, num_transformer_layers=3,
                 dim_feedforward=256, dropout=0.1, **kw):
        super().__init__()
        layers, in_ch = [], input_channels
        for i in range(n_cnn_blocks):
            out_ch = cnn_filters * (2**i)
            layers.append(nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size, padding=kernel_size//2),
                nn.BatchNorm1d(out_ch), nn.ReLU(True), nn.MaxPool1d(2, 2)))
            in_ch = out_ch
        self.cnn = nn.Sequential(*layers)
        self.proj = nn.Linear(in_ch, d_model)
        pe = torch.zeros(5000, d_model)
        pos = torch.arange(5000).float().unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.pe_drop = nn.Dropout(dropout)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model))
        layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, activation='gelu')
        self.transformer = nn.TransformerEncoder(layer, num_transformer_layers)
        self.fc = nn.Sequential(nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, num_classes))
    def forward(self, x):
        x = self.proj(self.cnn(x).permute(0, 2, 1))
        x = torch.cat([self.cls_token.expand(x.size(0), -1, -1), x], 1)
        x = self.pe_drop(x + self.pe[:, :x.size(1)])
        return self.fc(self.transformer(x)[:, 0])

# ── Model Registry ──
MODEL_REGISTRY = {
    'cnn_1d': CNN1D, 'leadwise_cnn': LeadwiseCNN, 'resnet': ResNet1D,
    'pretrained_resnet': PretrainedResNet1D, 'lstm': LSTMModel,
    'transformer': TransformerModel, 'cnn_lstm': CNNLSTM, 'cnn_transformer': CNNTransformer,
}

print(f"Registered {len(MODEL_REGISTRY)} models: {list(MODEL_REGISTRY.keys())}")

## 6. Training Engine

In [ ]:
# ── Loss ──
class WeightedBCEWithLogitsLoss(nn.Module):
    def __init__(self, class_weights=None):
        super().__init__()
        self.register_buffer('class_weights', class_weights)
    def forward(self, logits, targets):
        loss = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        if self.class_weights is not None:
            loss = loss * self.class_weights.unsqueeze(0)
        return loss.mean()

# ── Early Stopping ──
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.001):
        self.patience, self.min_delta = patience, min_delta
        self.best_score, self.counter, self.best_state = None, 0, None
    def __call__(self, score, model):
        if self.best_score is None or score < self.best_score - self.min_delta:
            self.best_score, self.counter = score, 0
            self.best_state = {k: v.clone() for k, v in model.state_dict().items()}
            return False
        self.counter += 1
        return self.counter >= self.patience
    def restore(self, model):
        if self.best_state:
            model.load_state_dict(self.best_state)

# ── Train / Validate ──
def train_one_epoch(model, loader, criterion, optimizer, scaler, use_amp, grad_clip=1.0):
    model.train()
    total, n = 0.0, 0
    for signals, labels in loader:
        signals, labels = signals.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if use_amp:
            with autocast():
                loss = criterion(model(signals), labels)
            scaler.scale(loss).backward()
            if grad_clip > 0: scaler.unscale_(optimizer); nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer); scaler.update()
        else:
            loss = criterion(model(signals), labels)
            loss.backward()
            if grad_clip > 0: nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
        total += loss.item(); n += 1
    return total / max(n, 1)

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total, n, all_logits, all_labels = 0.0, 0, [], []
    for signals, labels in loader:
        signals, labels = signals.to(DEVICE), labels.to(DEVICE)
        logits = model(signals)
        total += criterion(logits, labels).item(); n += 1
        all_logits.append(logits.cpu()); all_labels.append(labels.cpu())
    return total / max(n, 1), torch.cat(all_logits), torch.cat(all_labels)

def evaluate_model(model, loader, criterion, label_classes):
    _, logits, labels = validate(model, loader, criterion)
    probs = torch.sigmoid(logits).numpy()
    labels = labels.numpy()
    # Optimal thresholds
    thresholds = np.zeros(len(label_classes))
    for c in range(len(label_classes)):
        best_f1, best_t = 0, 0.5
        for t in np.arange(0.1, 0.9, 0.05):
            f1 = f1_score(labels[:, c], (probs[:, c] >= t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, t
        thresholds[c] = best_t
    preds = (probs >= thresholds).astype(np.float32)
    metrics = {
        'subset_accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro', zero_division=0),
        'f1_micro': f1_score(labels, preds, average='micro', zero_division=0),
        'precision_macro': precision_score(labels, preds, average='macro', zero_division=0),
        'recall_macro': recall_score(labels, preds, average='macro', zero_division=0),
        'roc_auc_macro': roc_auc_score(labels, probs, average='macro'),
    }
    return metrics, probs, preds, labels, thresholds

## 7. Train All Models

In [ ]:
# ── Build DataLoaders (shared across models to save memory) ──

def make_loaders(batch_size, num_workers=2):
    train_loader = DataLoader(ECGDataset(signals[train_idx], label_matrix[train_idx]),
                              batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader = DataLoader(ECGDataset(signals[val_idx], label_matrix[val_idx]),
                            batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(ECGDataset(signals[test_idx], label_matrix[test_idx]),
                             batch_size=batch_size, shuffle=False,
                             num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

In [ ]:
# ══════════════════════════════════════════════════════════════
#  MAIN TRAINING LOOP — trains all models one by one
# ══════════════════════════════════════════════════════════════

all_results = {}
models_dir = Path("outputs/models")
models_dir.mkdir(parents=True, exist_ok=True)

for model_idx, model_name in enumerate(MODELS_TO_TRAIN):
    print(f"\n{'='*70}")
    print(f"  [{model_idx+1}/{len(MODELS_TO_TRAIN)}] Training: {model_name}")
    print(f"{'='*70}")
    
    mcfg = MODEL_CONFIGS[model_name]
    
    # Seed
    random.seed(42); np.random.seed(42); torch.manual_seed(42)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(42)
    
    # Build model
    kwargs = {'input_channels': 12, 'num_classes': 5, **mcfg['params']}
    model = MODEL_REGISTRY[model_name](**kwargs).to(DEVICE)
    
    # Multi-GPU
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    
    params = sum(p.numel() for p in model.parameters())
    print(f"  Params: {params:,} | Batch: {mcfg['batch_size']} | LR: {mcfg['lr']} | Epochs: {mcfg['epochs']}")
    
    # DataLoaders
    train_loader, val_loader, test_loader = make_loaders(mcfg['batch_size'])
    
    # Loss, optimizer, scheduler
    cw = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)
    criterion = WeightedBCEWithLogitsLoss(cw).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=mcfg['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=mcfg['epochs'], eta_min=1e-5)
    
    use_amp = torch.cuda.is_available()
    scaler = GradScaler() if use_amp else None
    early_stopping = EarlyStopping(patience=mcfg['patience'])
    
    # Training
    history = {'train_loss': [], 'val_loss': [], 'lr': []}
    best_val = float('inf')
    
    for epoch in range(mcfg['epochs']):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, scaler, use_amp)
        val_loss, _, _ = validate(model, val_loader, criterion)
        
        lr = optimizer.param_groups[0]['lr']
        scheduler.step()
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['lr'].append(lr)
        
        print(f"  Epoch {epoch+1:>3}/{mcfg['epochs']} | loss: {train_loss:.4f} | val: {val_loss:.4f} | best: {best_val:.4f} | lr: {lr:.1e}")
        
        if val_loss < best_val:
            best_val = val_loss
            state = (model.module if hasattr(model, 'module') else model).state_dict()
            torch.save({'epoch': epoch, 'model_state_dict': state, 'val_loss': val_loss},
                       models_dir / f'best_{model_name}.pt')
        
        if early_stopping(val_loss, model):
            print(f"  ⛔ Early stopping at epoch {epoch+1}. Best: {best_val:.4f}")
            break
    
    # Restore best
    early_stopping.restore(model)
    
    # Save final
    state = (model.module if hasattr(model, 'module') else model).state_dict()
    torch.save({'epoch': epoch, 'model_state_dict': state, 'val_loss': val_loss},
               models_dir / f'final_{model_name}.pt')
    
    # ── Evaluate on test set ──
    base_model = model.module if hasattr(model, 'module') else model
    ckpt = torch.load(models_dir / f'best_{model_name}.pt', map_location=DEVICE)
    base_model.load_state_dict(ckpt['model_state_dict'])
    
    metrics, probs, preds, labels_np, thresholds = evaluate_model(model, test_loader, criterion, label_classes)
    all_results[model_name] = {'metrics': metrics, 'history': history, 'best_val': best_val}
    
    # Save results
    res_dir = Path(f"outputs/results/{model_name}")
    res_dir.mkdir(parents=True, exist_ok=True)
    with open(res_dir / 'metrics.json', 'w') as f:
        json.dump({k: float(v) for k, v in metrics.items()}, f, indent=2)
    np.save(res_dir / 'probabilities.npy', probs)
    np.save(res_dir / 'predictions.npy', preds)
    np.save(res_dir / 'labels.npy', labels_np)
    np.save(res_dir / 'optimal_thresholds.npy', thresholds)
    np.save(res_dir / 'history_train_loss.npy', history['train_loss'])
    np.save(res_dir / 'history_val_loss.npy', history['val_loss'])
    
    # Print metrics
    print(f"\n  Results for {model_name}:")
    print(f"    F1 Macro:   {metrics['f1_macro']:.4f}")
    print(f"    ROC-AUC:    {metrics['roc_auc_macro']:.4f}")
    print(f"    Accuracy:   {metrics['subset_accuracy']:.4f}")
    
    # Cleanup GPU memory
    del model, optimizer, scheduler, criterion, scaler
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n{'='*70}")
print(f"  ✅ ALL {len(MODELS_TO_TRAIN)} MODELS TRAINED SUCCESSFULLY")
print(f"{'='*70}")

## 8. Model Comparison

In [ ]:
# ── Comparison table ──
print(f"\n{'Model':<22} {'F1 Macro':>10} {'ROC-AUC':>10} {'Subset Acc':>12} {'Best Val':>10}")
print("-" * 66)

best_model, best_f1 = None, 0
for name in MODELS_TO_TRAIN:
    if name not in all_results:
        continue
    m = all_results[name]['metrics']
    bv = all_results[name]['best_val']
    marker = ""
    if m['f1_macro'] > best_f1:
        best_f1, best_model = m['f1_macro'], name
    print(f"{name:<22} {m['f1_macro']:>10.4f} {m['roc_auc_macro']:>10.4f} {m['subset_accuracy']:>12.4f} {bv:>10.4f}")

print("-" * 66)
print(f"🏆 Best model: {best_model} (F1 Macro: {best_f1:.4f})")

## 9. Training History Plots

In [ ]:
import matplotlib.pyplot as plt

n_models = len(all_results)
fig, axes = plt.subplots(n_models, 1, figsize=(14, 4 * n_models))
if n_models == 1: axes = [axes]

for ax, name in zip(axes, MODELS_TO_TRAIN):
    if name not in all_results: continue
    h = all_results[name]['history']
    ax.plot(h['train_loss'], label='Train', linewidth=2)
    ax.plot(h['val_loss'], label='Val', linewidth=2)
    ax.set_title(f'{name} — Loss', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_dir = Path("outputs/figures")
fig_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(fig_dir / "all_training_histories.png", dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved to {fig_dir / 'all_training_histories.png'}")

## 10. Download Outputs

In [ ]:
# Save comparison summary
summary = {name: all_results[name]['metrics'] for name in MODELS_TO_TRAIN if name in all_results}
with open("outputs/model_comparison.json", 'w') as f:
    json.dump(summary, f, indent=2)

shutil.make_archive('outputs_archive', 'zip', '.', 'outputs')
print("📥 Download: outputs_archive.zip")
print("📂 Contains: models, results, figures for all trained models")
print()
print("To use locally, extract into your project's outputs/ folder.")